# Inspect converted modules

## Inspect modules

In [1]:
import polars as pl
import polars_bio as pb
import sqlite3
from pathlib import Path
from pycomfort import files

from platformdirs import user_cache_dir
from pyfaidx import Fasta

# Configure Polars to show more rows and columns
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Increase table width
pl.Config.set_fmt_str_lengths(1000)  # Show longer string values without truncation

def list_tables(conn: sqlite3.Connection):
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    return tables

def print_tables(conn: sqlite3.Connection):
    tables = list_tables(conn)
    cursor = conn.cursor()
    # Get schema for each table
    for table in tables:
        table_name = table[0]
        print(f"\nSchema for table '{table_name}':")
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        for col in columns:
            print(f"  {col[1]} ({col[2]})")
    return tables

/home/antonkulaga/sources/prepare-annotations/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from os import listdir
from prepare_annotations.core.paths import (
    get_cache_dir,
    get_ensembl_cache,
    get_ensembl_variations_cache,
    get_ensembl_genome_cache,
    list_ensembl_variation_parquets,
    list_ensembl_genome_fastas,
    find_ensembl_genome_fasta,
)

# Get cache directories using the new helpers
cache = get_cache_dir()
ensembl_cache = get_ensembl_cache()  # homo_sapiens by default
ensembl_variations = get_ensembl_variations_cache()
ensembl_genome = get_ensembl_genome_cache()
clinvar_cache = cache / "clinvar"
dbsnp_t2t_cache = cache / "dbsnp"

print(f"Cache root: {cache}")
print(f"Ensembl cache: {ensembl_cache}")
print(f"Ensembl variations: {ensembl_variations}")
print(f"Ensembl genome: {ensembl_genome}")
print(f"\nVariation parquets available:")
for p in list_ensembl_variation_parquets()[:5]:
    print(f"  {p.name}")
print(f"  ... (total: {len(list_ensembl_variation_parquets())} files)")

print(f"\nGenome FASTAs available:")
for p in list_ensembl_genome_fastas():
    print(f"  {p.name}")

Cache root: /home/antonkulaga/.cache/just-dna-pipelines
Ensembl cache: /home/antonkulaga/.cache/just-dna-pipelines/ensembl/homo_sapiens
Ensembl variations: /home/antonkulaga/.cache/just-dna-pipelines/ensembl/homo_sapiens
Ensembl genome: /home/antonkulaga/.cache/just-dna-pipelines/ensembl/homo_sapiens/fasta/dna

Variation parquets available:
  homo_sapiens-chr1.vcf.parquet
  homo_sapiens-chr10.vcf.parquet
  homo_sapiens-chr11.vcf.parquet
  homo_sapiens-chr12.vcf.parquet
  homo_sapiens-chr13.vcf.parquet
  ... (total: 20 files)

Genome FASTAs available:
  Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz


## Loaded conversions

In [4]:
from pycomfort import files
from pathlib import Path
base = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = base / "data"
modules = data / "modules"
output = data / "output"
output_modules = output / "modules"
files.tprint(modules)

modules
	.gitignore
	just_drugs
		annotation_tab.tsv
	just_longevitymap
		longevitymap.sqlite
	just_coronary
		coronary.sqlite
	just_lipidmetabolism
		lipid_metabolism.sqlite
	just_vo2max
		vo2max.sqlite
	longevitymap_annotations.parquet
	just_superhuman
		superhuman.sqlite
	just_prs
		prs.sqlite
	just_cancer
		genes.txt


In [5]:
from pycomfort.files import tprint
tprint(output_modules)

modules
	longevitymap_annotations.parquet
	longevitymap
		annotations.parquet
		weights.parquet
		studies.parquet
	drugs
		annotations.parquet
		weights.parquet
		studies.parquet
	vo2max
		annotations.parquet
		weights.parquet
		studies.parquet
	lipidmetabolism
		annotations.parquet
		weights.parquet
		studies.parquet
	coronary
		annotations.parquet
		weights.parquet
		studies.parquet
	superhuman
		annotations.parquet
		weights.parquet
		studies.parquet


In [6]:
# LongevityMap
longevitymap_annotations = pl.scan_parquet(output_modules / "longevitymap" / "annotations.parquet")
longevitymap_weights = pl.scan_parquet(output_modules / "longevitymap" / "weights.parquet")
longevitymap_studies = pl.scan_parquet(output_modules / "longevitymap" / "studies.parquet")

# Other modules examples:
# coronary_weights = pl.scan_parquet(output_modules / "coronary" / "weights.parquet")
# drugs_annotations = pl.scan_parquet(output_modules / "drugs" / "annotations.parquet")
# lipidmetabolism_weights = pl.scan_parquet(output_modules / "lipidmetabolism" / "weights.parquet")
# vo2max_weights = pl.scan_parquet(output_modules / "vo2max" / "weights.parquet")
# superhuman_annotations = pl.scan_parquet(output_modules / "superhuman" / "annotations.parquet")

In [26]:
longevitymap_weights.head(5).collect(engine="streaming")

rsid,genotype,module,weight,state,priority,conclusion,curator,method
str,list[str],str,f64,str,str,str,str,str
"""rs9472817""","[""C"", ""C""]","""longevitymap""",0.49,"""protective""","""0.49""",null,"""Olga Borysova""","""literature_review"""
"""rs10229977""","[""?"", ""C""]","""longevitymap""",0.07,"""protective""","""0.14""",null,"""Olga Borysova""","""literature_review"""
"""rs3906146""","[""C"", ""C""]","""longevitymap""",0.2,"""protective""","""0.2""",null,"""Olga Borysova""","""literature_review"""
"""rs2901840""","[""?"", ""A""]","""longevitymap""",0.07,"""protective""","""0.14""",null,"""Olga Borysova""","""literature_review"""
"""rs923520""","[""?"", ""C""]","""longevitymap""",0.07,"""protective""","""0.14""",null,"""Olga Borysova""","""literature_review"""


In [8]:
longevitymap_weights.select("genotype").unique().collect(engine="streaming")

genotype
list[str]
"[""C"", ""C""]"
"[""G"", ""G""]"
"[""?"", ""A""]"
"[""C"", ""G""]"
"[""?"", ""C""]"
"[""?"", ""G""]"
"[""?"", ""T""]"
"[""A"", ""C""]"
"[""C"", ""T""]"


In [7]:
longevitymap_weights.count().collect(engine="streaming")

rsid,genotype,module,weight,state,priority,conclusion,curator,method
u32,u32,u32,u32,u32,u32,u32,u32,u32
1043,1043,1043,1043,1043,1043,0,1043,1043


## Load weights from original longevitymap SQLite

In [28]:
# Connect to the database
db_path = modules / "just_longevitymap" / "longevitymap.sqlite"
conn = sqlite3.connect(db_path)

print_tables(conn)


Schema for table 'gene':
  id (INTEGER)
  name (TEXT)
  symbol (TEXT)
  alias (TEXT)
  description (TEXT)
  omim (TEXT)
  ensembl (TEXT)
  uniprot (TEXT)
  unigene (TEXT)
  cytogenetic_location (TEXT)

Schema for table 'population':
  id (INTEGER)
  name (TEXT)

Schema for table 'variant':
  id (INTEGER)
  location (TEXT)
  study_design (TEXT)
  conclusions (TEXT)
  association (TEXT)
  gender (TEXT)
  quickref (TEXT)
  quickyear (INTEGER)
  quickpubmed (TEXT)
  identifier (TEXT)
  gene_id (INTEGER)
  population_id (INTEGER)

Schema for table 'allele_weights':
  id (INTEGER)
  allele (TEXT)
  state (TEXT)
  zygosity (TEXT)
  weight (REAL)
  rsid (TEXT)
  priority (TEXT)
  category_id (INTEGER)

Schema for table 'categories':
  id (INTEGER)
  name (TEXT)


[('gene',),
 ('population',),
 ('variant',),
 ('allele_weights',),
 ('categories',)]

In [29]:
sqlite_weights = pl.read_database("""
SELECT rsid, allele, state, zygosity, weight, priority, categories.name
FROM allele_weights, categories WHERE categories.id = allele_weights.category_id
""", connection=conn).lazy()

sqlite_variants = pl.read_database("""
SELECT variant.identifier as rsid, variant.study_design, variant.conclusions, variant.association, 
       variant.gender, variant.quickref, variant.quickyear, variant.quickpubmed, 
       population.name as population_name
FROM variant
JOIN population ON variant.population_id = population.id
""", connection=conn).with_columns(
    (pl.col("association") == "significant").alias("is_significant")
).drop("association").lazy()

sqlite_extended_weights = sqlite_weights.join(sqlite_variants, on="rsid")
sqlite_extended_weights.head(5).collect(engine="streaming")

rsid,allele,state,zygosity,weight,priority,name,study_design,conclusions,gender,quickref,quickyear,quickpubmed,population_name,is_significant
str,str,str,str,f64,str,str,str,str,str,str,i64,str,str,bool
"""rs662""","""T""","""ref""","""hom""",0.31,"""0.62""","""lipids""","""A Leucine (L allele) to Methionine (M allele) substitution at codon 55, and a Glutamine (A allele) to Arginine (B allele) substitution at codon 192 were examined in 579 people aged 20 to 65 years old, and in 308 centenarians""","""Significant differences between young people and centenarians were observed. The percentage of carriers of the B allele at codon 192 is higher in centenarians than in controls (0.539 vs 0.447), though this was due to an increase of people carrying M alleles at codon 55.""","""male/female""","""Bonafè et al. (2002)""",2002,"""12082503""","""Italian""",true
"""rs1800896""","""C""","""alt""","""het""",0.25,"""0.5""","""inflammation""","""The -1082G/A, -819C/T and -592C/A proximal promoter SNPs were examined in 190 centenarians (>99 years old, 159 women and 31 men) and in 260 control subjects (99 women and 161 men less than 60 years old)""","""The -1082G homozygous genotype, associated with high IL-10 production, was increased in centenarian men but not in centenarian women. No difference was found between centenarians and control subjects regarding the other two SNPs.""","""male/female""","""Lio et al. (2002)""",2002,"""11857058""","""Italian""",true
"""rs1800896""","""C""","""alt""","""hom""",0.5,"""0.5""","""inflammation""","""The -1082G/A, -819C/T and -592C/A proximal promoter SNPs were examined in 190 centenarians (>99 years old, 159 women and 31 men) and in 260 control subjects (99 women and 161 men less than 60 years old)""","""The -1082G homozygous genotype, associated with high IL-10 production, was increased in centenarian men but not in centenarian women. No difference was found between centenarians and control subjects regarding the other two SNPs.""","""male/female""","""Lio et al. (2002)""",2002,"""11857058""","""Italian""",true
"""rs1800896""","""C""","""alt""","""het""",0.25,"""0.5""","""inflammation""","""-1082 G/A SNP was examined in 72 centenarian men, 102 centenarian women and healthy unrelated controls (115 men and 112 women, aged 22-60 years)""","""The number of male centenarians homozygous for the -1082G genotype, suggested to be associated with high IL-10 production, was significantly increased in comparison with younger control subjects. No significant differences were observed between women and controls.""","""male/female""","""Lio et al. (2003)""",2003,"""12676903""","""Italian""",true
"""rs1800896""","""C""","""alt""","""hom""",0.5,"""0.5""","""inflammation""","""-1082 G/A SNP was examined in 72 centenarian men, 102 centenarian women and healthy unrelated controls (115 men and 112 women, aged 22-60 years)""","""The number of male centenarians homozygous for the -1082G genotype, suggested to be associated with high IL-10 production, was significantly increased in comparison with younger control subjects. No significant differences were observed between women and controls.""","""male/female""","""Lio et al. (2003)""",2003,"""12676903""","""Italian""",true


In [30]:
sqlite_extended_weights.join(longevitymap_weights, on="rsid", suffix="_longevitymap").head(5).collect(engine="streaming")

rsid,allele,state,zygosity,weight,priority,name,study_design,conclusions,gender,quickref,quickyear,quickpubmed,population_name,is_significant,genotype,module,weight_longevitymap,state_longevitymap,priority_longevitymap,conclusion,curator,method
str,str,str,str,f64,str,str,str,str,str,str,i64,str,str,bool,list[str],str,f64,str,str,str,str,str
"""rs2075650""","""G""","""alt""","""het""",-0.195,"""0.39""","""mitochondria""","""Genome-wide association study in 403 unrelated nonagenarians from long-living families and 1670 younger controls. Strongest candidates were then investigated in a meta-analysis of 4149 nonagenarian cases and 7582 younger controls.""","""No SNP reached significance in the GWAS but 62 SNPs had an indicative association with survival into old age. Of these 62 SNPs then studied in the meta-analysis, only one was significant: rs2075650 located in TOMM40 and close to APOE. This association may be due to linkage disequilibrium with rs429358 and rs7412 in APOE; both rs429358 and rs7412 were associated with longevity in the meta-analysis.""","""male/female""","""Deelen et al. (2011)""",2011,"""21418511""","""Dutch""",true,"[""G"", ""G""]","""longevitymap""",-0.39,"""risk""","""0.39""",null,"""Olga Borysova""","""literature_review"""
"""rs2075650""","""G""","""alt""","""het""",-0.195,"""0.39""","""mitochondria""","""Genome-wide association study in 403 unrelated nonagenarians from long-living families and 1670 younger controls. Strongest candidates were then investigated in a meta-analysis of 4149 nonagenarian cases and 7582 younger controls.""","""No SNP reached significance in the GWAS but 62 SNPs had an indicative association with survival into old age. Of these 62 SNPs then studied in the meta-analysis, only one was significant: rs2075650 located in TOMM40 and close to APOE. This association may be due to linkage disequilibrium with rs429358 and rs7412 in APOE; both rs429358 and rs7412 were associated with longevity in the meta-analysis.""","""male/female""","""Deelen et al. (2011)""",2011,"""21418511""","""Dutch""",true,"[""?"", ""G""]","""longevitymap""",-0.195,"""risk""","""0.39""",null,"""Olga Borysova""","""literature_review"""
"""rs2075650""","""G""","""alt""","""hom""",-0.39,"""0.39""","""mitochondria""","""Genome-wide association study in 403 unrelated nonagenarians from long-living families and 1670 younger controls. Strongest candidates were then investigated in a meta-analysis of 4149 nonagenarian cases and 7582 younger controls.""","""No SNP reached significance in the GWAS but 62 SNPs had an indicative association with survival into old age. Of these 62 SNPs then studied in the meta-analysis, only one was significant: rs2075650 located in TOMM40 and close to APOE. This association may be due to linkage disequilibrium with rs429358 and rs7412 in APOE; both rs429358 and rs7412 were associated with longevity in the meta-analysis.""","""male/female""","""Deelen et al. (2011)""",2011,"""21418511""","""Dutch""",true,"[""G"", ""G""]","""longevitymap""",-0.39,"""risk""","""0.39""",null,"""Olga Borysova""","""literature_review"""
"""rs2075650""","""G""","""alt""","""hom""",-0.39,"""0.39""","""mitochondria""","""Genome-wide association study in 403 unrelated nonagenarians from long-living families and 1670 younger controls. Strongest candidates were then investigated in a meta-analysis of 4149 nonagenarian cases and 7582 younger controls.""","""No SNP reached significance in the GWAS but 62 SNPs had an indicative association with survival into old age. Of these 62 SNPs then studied in the meta-analysis, only one was significant: rs2075650 located in TOMM40 and close to APOE. This association may be due to linkage disequilibrium with rs429358 and rs7412 in APOE; both rs429358 and rs7412 were associated with longevity in the meta-analysis.""","""male/female""","""Deelen et al. (2011)""",2011,"""21418511""","""Dutch""",true,"[""?"", ""G""]","""longevitymap""",-0.195,"""risk""","""0.39""",null,"""Olga Borysova""",

In [10]:
sqlite_weights.head(5).collect(engine="streaming")

rsid,allele,state,zygosity,weight,priority,name
str,str,str,str,f64,str,str
"""rs7412""","""T""","""alt""","""het""",0.5,"""1.0""","""lipids"""
"""rs7412""","""T""","""alt""","""hom""",1.0,"""1.0""","""lipids"""
"""rs429358""","""C""","""alt""","""het""",-0.5,"""1.0""","""lipids"""
"""rs429358""","""C""","""alt""","""hom""",-1.0,"""1.0""","""lipids"""
"""rs5882""","""G""","""ref""","""hom""",0.97,"""0.97""","""lipids"""


## Reading Ensembl Genome FASTA

Using `polars_bio` to read the downloaded genome FASTA files.

In [33]:
from prepare_annotations.core.paths import (
    find_ensembl_genome_fasta,
    list_ensembl_genome_fastas,
)


def _resolve_uncompressed_fasta_path(fasta_path: Path) -> Path:
    """Our downloader stores *.fa.gz and (when --index) also creates *.fa next to it."""
    if fasta_path.suffix == ".gz":
        return fasta_path.with_suffix("")
    return fasta_path


def _normalize_contig_name(fa: Fasta, chrom: str) -> str:
    """Accept '21' or 'chr21' depending on what the FASTA contains."""
    keys = set(fa.keys())
    if chrom in keys:
        return chrom
    if chrom.startswith("chr") and chrom[3:] in keys:
        return chrom[3:]
    if not chrom.startswith("chr") and f"chr{chrom}" in keys:
        return f"chr{chrom}"
    raise KeyError(f"Chromosome '{chrom}' not found in FASTA (examples: {sorted(list(keys))[:10]} ...)")


# Prefer a chromosome-specific FASTA if present; otherwise use primary assembly.
# NOTE: find_ensembl_genome_fasta() currently only finds *.fa.gz in the cache.
chr21_gz = find_ensembl_genome_fasta(genome_type="chromosome", chromosome="21")
if chr21_gz is None:
    chr21_gz = find_ensembl_genome_fasta(genome_type="primary_assembly")

print("Available genome FASTAs:")
for p in list_ensembl_genome_fastas():
    print(" ", p.name)

print(f"\nSelected FASTA (downloaded): {chr21_gz}")
if chr21_gz is None:
    raise FileNotFoundError(
        "No Ensembl genome FASTA found in cache. Download it with: uv run prepare-annotations genome"
    )

chr21_fa = _resolve_uncompressed_fasta_path(chr21_gz)
print(f"Uncompressed FASTA (for pyfaidx): {chr21_fa}")
print(f"FASTA index expected at: {chr21_fa}.fai")

# pyfaidx requires the uncompressed fasta + .fai for fast random access.
# If the .fa doesn't exist yet, re-run with indexing enabled:
#   uv run prepare-annotations genome --index
fa = Fasta(str(chr21_fa), as_raw=True, sequence_always_upper=True)


Available genome FASTAs:
  Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

Selected FASTA (downloaded): /home/antonkulaga/.cache/just-dna-pipelines/ensembl/homo_sapiens/fasta/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
Uncompressed FASTA (for pyfaidx): /home/antonkulaga/.cache/just-dna-pipelines/ensembl/homo_sapiens/fasta/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa
FASTA index expected at: /home/antonkulaga/.cache/just-dna-pipelines/ensembl/homo_sapiens/fasta/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai


In [34]:
# Example: fetch a single nucleotide by genomic coordinate (1-based)
chrom = "21"
pos_1based = 1_000_000

contig = _normalize_contig_name(fa, chrom)
base = fa.get_seq(contig, pos_1based, pos_1based)
print(f"{contig}:{pos_1based} = {base.seq}")

# Example: fetch a window
window = fa.get_seq(contig, pos_1based - 10, pos_1based + 10)
print(f"{contig}:{pos_1based-10}-{pos_1based+10} = {window.seq}")

: 

In [ ]:
# Optional: vectorized lookup for a Polars table of coordinates
coords = pl.DataFrame(
    {
        "chrom": ["21", "21"],
        "pos_1based": [1_000_000, 1_000_001],
    }
)

coords_with_base = coords.with_columns(
    pl.struct(["chrom", "pos_1based"]).map_elements(
        lambda r: fa.get_seq(_normalize_contig_name(fa, r["chrom"]), r["pos_1based"], r["pos_1based"]).seq,
        return_dtype=pl.Utf8,
    ).alias("ref_base"),
)

coords_with_base

Primary assembly: None
Chromosome 21: None


In [ ]:
# Notes:
# - pyfaidx will automatically use `<fasta>.fai` if present.
# - For the full GRCh38 primary assembly, the uncompressed `.fa` is very large.
#   If you only need a few chromosomes, consider:
#     uv run prepare-annotations genome --type chromosome --chromosome 21
#   (and keep --index enabled).